# IHRAS — 00. Data setup: NIH ChestX-ray14 to Google Drive

Downloads the NIH ChestX-ray14 archive from Kaggle **once**, stores it in Google Drive, and records
its provenance (SHA-256 checksums, file counts, metadata checks). Later notebooks copy the data
from Drive instead of downloading it again.

**Runtime:** CPU is enough (no GPU units are spent). Disk needed: about 45 GB free in `/content`.

**Credentials:** a Colab secret named `KAGGLE_API_TOKEN` (Kaggle → Settings → API → new token).
Legacy alternatives: secrets `KAGGLE_USERNAME` + `KAGGLE_KEY`, or `kaggle.json` in `DRIVE_ROOT`.

**Output in Drive** (`DATA_DIR`):
```
data.zip                Kaggle archive of nih-chest-xrays/data (~42 GB)
meta/                   Data_Entry_2017.csv, BBox_List_2017.csv, train_val_list.txt, test_list.txt
DATA_MANIFEST.json      provenance: source, date, tool versions, checksums, counts, metadata checks
```

The notebook is idempotent: if `DATA_MANIFEST.json` already exists, it only re-validates.

## 1. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pesquisa/ihras2"   # project folder in Google Drive
DATA_DIR = f"{DRIVE_ROOT}/data/nih"                     # where the dataset is stored
KAGGLE_DATASET = "nih-chest-xrays/data"                 # Kaggle copy with the original 2017 metadata
LOCAL_DIR = "/content/nih_download"                     # local scratch area for the download
EXPECTED_IMAGES = 112_120                               # images in NIH ChestX-ray14
VERIFY_DRIVE_COPY_HASH = True   # re-hash the copy in Drive (slow, ~10 min) to prove it is intact

## 2. Mount Drive and define helpers

In [ ]:
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DATA_DIR}/meta", exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

MANIFEST = f"{DATA_DIR}/DATA_MANIFEST.json"
DRIVE_ZIP = f"{DATA_DIR}/data.zip"
META_FILES = ("Data_Entry_2017.csv", "Data_Entry_2017_v2020.csv", "BBox_List_2017.csv",
              "train_val_list.txt", "test_list.txt")


def utc_now():
    """Current UTC time as an ISO-8601 string (used in every provenance record)."""
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def sha256(path, chunk=1 << 22):
    """SHA-256 of a file, streamed in 4 MB chunks so that large archives fit in memory."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def free_gb(path):
    """Free disk space (GB) on the file system that holds `path`."""
    return shutil.disk_usage(path).free / 1e9

print("Drive data dir:", DATA_DIR)
print(f"free local disk: {free_gb('/content'):.0f} GB")

## 3. Kaggle credentials and client

In [ ]:
# A recent client is required: older versions do not understand the new single-string API token.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)

from google.colab import userdata


def secret(name):
    """Read a Colab secret; return None if it is missing or notebook access was not granted."""
    try:
        return userdata.get(name)
    except Exception:
        return None


if secret("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = secret("KAGGLE_API_TOKEN")
    CRED = "KAGGLE_API_TOKEN secret"
elif secret("KAGGLE_USERNAME") and secret("KAGGLE_KEY"):
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY")
    CRED = "legacy username/key secrets"
elif os.path.exists(f"{DRIVE_ROOT}/kaggle.json"):
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    CRED = "legacy kaggle.json"
else:
    raise RuntimeError("No Kaggle credentials: create the Colab secret KAGGLE_API_TOKEN "
                       "and enable notebook access to it.")

KAGGLE_VERSION = subprocess.run(["kaggle", "--version"], capture_output=True, text=True).stdout.strip()
print("credentials:", CRED, "|", KAGGLE_VERSION)

## 4. Download (skipped if the dataset is already in Drive)

The archive is downloaded to the local disk first and then copied to Drive in one sequential write,
which is faster and more reliable than writing a 42 GB stream directly to the Drive mount.

In [ ]:
already_done = os.path.exists(MANIFEST) and os.path.exists(DRIVE_ZIP)
if already_done:
    print("dataset already in Drive; downloading is skipped:", DRIVE_ZIP)
else:
    assert free_gb("/content") > 45, "not enough local disk for the ~42 GB archive"
    t0 = datetime.datetime.now()
    # The client prints its own progress bar; check=True stops the notebook on failure.
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", LOCAL_DIR],
                   check=True)
    zips = [f for f in os.listdir(LOCAL_DIR) if f.endswith(".zip")]
    assert len(zips) == 1, f"expected one archive, found {zips}"
    LOCAL_ZIP = f"{LOCAL_DIR}/{zips[0]}"
    DOWNLOAD_SECONDS = (datetime.datetime.now() - t0).total_seconds()
    print(f"downloaded {os.path.getsize(LOCAL_ZIP) / 1e9:.1f} GB in {DOWNLOAD_SECONDS / 60:.0f} min")

## 5. Validate the archive and extract the metadata files

In [ ]:
ZIP_TO_READ = DRIVE_ZIP if already_done else LOCAL_ZIP

with zipfile.ZipFile(ZIP_TO_READ) as zf:
    names = zf.namelist()
    pngs = [n for n in names if n.lower().endswith(".png")]
    folders = sorted({n.split("/")[0] for n in pngs})
    # Copy each metadata file to meta/ under its plain file name, whatever its path in the zip.
    found_meta = []
    for info in zf.infolist():
        base = os.path.basename(info.filename)
        if base in META_FILES:
            with zf.open(info) as src, open(f"{DATA_DIR}/meta/{base}", "wb") as dst:
                shutil.copyfileobj(src, dst)
            found_meta.append(base)

print(f"{len(names)} entries, {len(pngs)} PNG images in {len(folders)} folders: {folders[:3]} ...")
print("metadata extracted:", sorted(found_meta))
assert len(pngs) == EXPECTED_IMAGES, f"expected {EXPECTED_IMAGES} images, found {len(pngs)}"
assert "Data_Entry_2017.csv" in found_meta and "BBox_List_2017.csv" in found_meta

## 6. Metadata checks

The reproduced article used the **original** `Data_Entry_2017.csv`. Its signature is a handful of
impossible ages (> 120 years) that were corrected in the 2020 re-release (`ASSUMPTIONS.md`, A1).
The sample size (50,208) depends on this version, so it is checked before anything else runs.

In [ ]:
import pandas as pd

meta = pd.read_csv(f"{DATA_DIR}/meta/Data_Entry_2017.csv")
ages = pd.to_numeric(meta["Patient Age"].astype(str).str.extract(r"(\d+)")[0])
bbox = pd.read_csv(f"{DATA_DIR}/meta/BBox_List_2017.csv")
test_list = open(f"{DATA_DIR}/meta/test_list.txt").read().split()

checks = {
    "rows": int(len(meta)),
    "patients": int(meta["Patient ID"].nunique()),
    "age_max": int(ages.max()),
    "n_age_over_120": int((ages > 120).sum()),
    "bbox_rows": int(len(bbox)),
    "bbox_images": int(bbox["Image Index"].nunique()),
    "test_list_images": len(test_list),
}
checks["looks_like_original_2017"] = checks["n_age_over_120"] > 0
print(json.dumps(checks, indent=1))
assert checks["rows"] == EXPECTED_IMAGES, "unexpected number of metadata rows"
if not checks["looks_like_original_2017"]:
    print("WARNING: this looks like the 2020 metadata; the sample will not have 50,208 images.")

## 7. Store in Drive and write the provenance manifest

In [ ]:
if not already_done:
    local_hash = sha256(LOCAL_ZIP)
    print("local archive sha256:", local_hash)
    shutil.copyfile(LOCAL_ZIP, DRIVE_ZIP)
    assert os.path.getsize(DRIVE_ZIP) == os.path.getsize(LOCAL_ZIP), "size mismatch after copy"
    drive_hash = sha256(DRIVE_ZIP) if VERIFY_DRIVE_COPY_HASH else None
    if drive_hash:
        assert drive_hash == local_hash, "the Drive copy differs from the downloaded archive"
    manifest = {
        "source": f"kaggle:{KAGGLE_DATASET}",
        "downloaded_utc": utc_now(),
        "download_seconds": round(DOWNLOAD_SECONDS),
        "kaggle_client": KAGGLE_VERSION,
        "credentials": CRED,
        "archive": {"file": "data.zip", "bytes": os.path.getsize(DRIVE_ZIP),
                    "sha256": local_hash, "drive_copy_verified": drive_hash is not None},
        "png_images": len(pngs),
        "image_folders": folders,
        "metadata_files": {f: {"sha256": sha256(f"{DATA_DIR}/meta/{f}"),
                               "bytes": os.path.getsize(f"{DATA_DIR}/meta/{f}")}
                           for f in sorted(found_meta)},
        "checks": checks,
        "validations": [],
    }
else:
    manifest = json.load(open(MANIFEST))
    # Re-validation: the metadata in Drive must still match the recorded checksums.
    for f, rec in manifest["metadata_files"].items():
        assert sha256(f"{DATA_DIR}/meta/{f}") == rec["sha256"], f"{f} changed since download"

manifest["validations"].append({"utc": utc_now(), "png_images": len(pngs), "checks": checks})
json.dump(manifest, open(MANIFEST, "w"), indent=1)
print(json.dumps({k: manifest[k] for k in ("source", "downloaded_utc", "archive", "png_images")},
                 indent=1))

## 8. Free the local disk

In [ ]:
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
print("done. Dataset and provenance manifest in", DATA_DIR)